# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

The dataset is described by a Croissant schema and contains multiple entities, including distributions, file objects, record sets, and fields. In this notebook, we use `@id` values whenever referring to dataset entities, and we rely on variables so the workflow can be adapted to other Croissant datasets.

### Dataset Source
Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Notes
- The dataset metadata indicates a provenance-rich, multi-table food-health knowledge graph.
- Some datasets expose record sets directly in the Croissant metadata, while others primarily expose distributions and file objects. This notebook is written defensively so it can inspect the dataset structure first and then load available record sets if present.
- We avoid treating `dataset.metadata` like a dictionary directly. Instead, we access it as a metadata object and convert it to JSON only when needed.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn pyarrow

## 1. Data Loading
Load the Croissant dataset and inspect its high-level metadata.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON when needed
metadata_json = dataset.metadata.to_json()

dataset_name = metadata_json.get("name", "Unknown dataset")
dataset_description = metadata_json.get("description", "No description available.")
dataset_version = metadata_json.get("version", "Unknown")
dataset_license = metadata_json.get("license", "Unknown")
dataset_identifier = metadata_json.get("identifier", "Unknown")

print(f"Dataset: {dataset_name}")
print(f"Identifier: {dataset_identifier}")
print(f"Version: {dataset_version}")
print(f"License: {dataset_license}")
print("\nDescription:")
print(dataset_description)

In [ ]:
# Optional: inspect the top-level metadata keys available after JSON conversion
sorted(metadata_json.keys())

## 2. Metadata Overview
Review available record sets, distributions, and related entities. All references are shown using their `@id` values.

This is especially useful for multi-table datasets, because the next steps depend on knowing which record sets can be loaded.

In [ ]:
# Summarize key metadata sections using JSON-converted metadata
record_sets_json = metadata_json.get("recordSet", []) or []
distributions_json = metadata_json.get("distribution", []) or []
visualizations_json = metadata_json.get("visualization", []) or []

print(f"Number of record sets declared: {len(record_sets_json)}")
print(f"Number of distributions declared: {len(distributions_json)}")
print(f"Number of visualizations declared: {len(visualizations_json)}")

In [ ]:
# Display distribution @id values
distribution_ids = [item.get("@id") for item in distributions_json if isinstance(item, dict)]
pd.DataFrame({"distribution_@id": distribution_ids}).head(20)

In [ ]:
# Build a record-set summary table if record sets are present
record_set_rows = []

for rs in record_sets_json:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    rs_description = rs.get("description")
    fields = rs.get("field", []) or []
    field_ids = [field.get("@id") for field in fields if isinstance(field, dict)]
    record_set_rows.append({
        "record_set_@id": rs_id,
        "name": rs_name,
        "description": rs_description,
        "n_fields": len(field_ids),
        "field_@ids": field_ids
    })

record_set_summary_df = pd.DataFrame(record_set_rows)
record_set_summary_df

### Inspect record-level access
If record sets are available, the following cell shows a small sample from the first discovered record set using its `@id`.

In [ ]:
record_set_ids = record_set_summary_df["record_set_@id"].dropna().tolist() if not record_set_summary_df.empty else []

if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f"Sampling records from record set @id: {sample_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets were declared in metadata_json['recordSet']. This dataset may primarily expose distributions/file objects rather than Croissant record sets.")

## 3. Data Extraction
Load available record sets into pandas DataFrames.

We use record set `@id` values dynamically so the notebook remains general. If no record sets are present, the notebook will explain that and skip record extraction.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} rows from {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

In [ ]:
# Summarize extracted DataFrames
extraction_summary = []

for record_set_id, df in dataframes.items():
    extraction_summary.append({
        "record_set_@id": record_set_id,
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "columns": list(df.columns)
    })

extraction_summary_df = pd.DataFrame(extraction_summary)
extraction_summary_df

In [ ]:
# Preview the first non-empty extracted DataFrame, if available
selected_record_set_id = None

for rs_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rs_id
        break

if selected_record_set_id is not None:
    print(f"Selected record set for exploration: {selected_record_set_id}")
    print("Columns:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No non-empty record-set DataFrame was loaded. The remainder of the notebook contains defensive logic and may skip table-specific analysis.")

## 4. Exploratory Data Analysis (EDA)
In this section, we identify numeric fields automatically, choose one for simple processing, and demonstrate filtering, normalization, missing-value inspection, and grouping.

Because Croissant datasets can vary, the code below adapts to the first loaded non-empty table.

In [ ]:
if selected_record_set_id is not None:
    analysis_df = dataframes[selected_record_set_id].copy()
    
    # Try to convert object columns to numeric when possible
    for col in analysis_df.columns:
        if analysis_df[col].dtype == "object":
            analysis_df[col] = pd.to_numeric(analysis_df[col], errors="ignore")
    
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_like_columns = [
        col for col in analysis_df.columns
        if analysis_df[col].dtype == "object" or str(analysis_df[col].dtype).startswith("string")
    ]
    
    print(f"Numeric columns found: {numeric_columns}")
    print(f"Categorical/text-like columns found: {categorical_like_columns[:20]}")
else:
    analysis_df = pd.DataFrame()
    numeric_columns = []
    categorical_like_columns = []
    print("Skipping EDA because no table was available.")

In [ ]:
# Missing-value overview
if not analysis_df.empty:
    missing_summary_df = (
        analysis_df.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_pct=lambda d: 100 * d["missing_count"] / len(analysis_df))
        .sort_values("missing_count", ascending=False)
    )
    display(missing_summary_df.head(15))
else:
    print("No missing-value summary available.")

In [ ]:
# Demonstrate filtering and normalization on a numeric field, if one exists
if not analysis_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    numeric_series = analysis_df[numeric_field_id].dropna()
    threshold = numeric_series.quantile(0.5) if not numeric_series.empty else 0
    
    filtered_df = analysis_df[analysis_df[numeric_field_id] > threshold].copy()
    
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Threshold (median-based): {threshold}")
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        normalized_field_id = f"{numeric_field_id}_normalized"
        filtered_df[normalized_field_id] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
        print(f"Normalized field created: {normalized_field_id}")
        display(filtered_df[[numeric_field_id, normalized_field_id]].head())
    else:
        normalized_field_id = None
        print("Standard deviation is zero or undefined; skipping normalization.")
else:
    filtered_df = pd.DataFrame()
    numeric_field_id = None
    normalized_field_id = None
    print("No numeric field available for filtering and normalization.")

In [ ]:
# Group by a text-like field if available
if not filtered_df.empty and categorical_like_columns and numeric_field_id is not None:
    group_field_id = categorical_like_columns[0]
    grouped_df = (
        filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("count", ascending=False)
    )
    print(f"Grouped by field: {group_field_id}")
    display(grouped_df.head(10))
else:
    group_field_id = None
    grouped_df = pd.DataFrame()
    print("No suitable grouping field was available.")

### Outlier handling example
A common preprocessing step is to remove extreme values using the interquartile range (IQR). The next cell demonstrates that pattern if a numeric field is available.

In [ ]:
if not analysis_df.empty and numeric_field_id is not None:
    q1 = analysis_df[numeric_field_id].quantile(0.25)
    q3 = analysis_df[numeric_field_id].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    iqr_filtered_df = analysis_df[
        analysis_df[numeric_field_id].between(lower_bound, upper_bound, inclusive="both")
    ].copy()
    
    print(f"Numeric field: {numeric_field_id}")
    print(f"IQR bounds: [{lower_bound}, {upper_bound}]")
    print(f"Original rows: {len(analysis_df)}")
    print(f"Rows after IQR filtering: {len(iqr_filtered_df)}")
else:
    iqr_filtered_df = pd.DataFrame()
    print("Skipping outlier handling example because no numeric field was found.")

## 5. Visualization
Visualize the distribution of a numeric field and, when possible, the top grouped categories.

In [ ]:
if not analysis_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(10, 5))
    sns.histplot(analysis_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}\nRecord set @id: {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for histogram visualization.")

In [ ]:
if not grouped_df.empty and group_field_id is not None:
    top_grouped_df = grouped_df.head(10).reset_index()
    
    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_grouped_df, x=group_field_id, y="count")
    plt.title(f"Top categories by count for {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No grouped data available for categorical visualization.")

## 6. Practical Notes for This Dataset
The FAIR²-FoodHealth metadata describes a mixed dataset with synthetic, curated, empirical, and provenance-oriented content.

When working with loaded tables, keep these points in mind:
- Use explicit markers such as review or quality fields when present.
- Do not interpret synthetic or navigation-oriented records as observational evidence.
- Keep source-derived, curated, and synthetic content separated in downstream analysis.
- Respect the semantics of missingness, since missing values may mean not applicable, unresolved, or unavailable rather than random absence.
- Use `@id` values for robust linkage across tables and entities.

## 7. Conclusion
In this notebook, we:
- Loaded the dataset from its Croissant schema URL using `mlcroissant`.
- Inspected high-level metadata and listed dataset entities by `@id`.
- Attempted to load available record sets dynamically.
- Built pandas DataFrames for exploration.
- Demonstrated basic preprocessing steps including missing-value inspection, filtering, normalization, grouping, and outlier handling.
- Visualized available numeric and grouped data.

This workflow is designed to be reusable for other Croissant datasets, especially those with multiple record sets and entity identifiers.